# VLM Comparison: Thermal Image Analysis of Industrial Motors

This notebook compares two Vision-Language Models on thermal images of induction motors from the [Mendeley dataset](https://data.mendeley.com/datasets/m4sbt8hbvk/3):

| Model | Parameters | Quantization |
|---|---|---|
| **Qwen2.5-VL-7B-Instruct** | 7B | 4-bit (bitsandbytes) |
| **Llama-3.2-Vision-11B-Instruct** | 11B | 4-bit (bitsandbytes) |

**Workflow:**
1. Load and run Qwen on all images -> save results -> unload
2. Load and run Llama on all images -> save results -> unload
3. Compare outputs side-by-side

**Hardware target:** NVIDIA L4 (24 GB VRAM)

**Prerequisites:**
- Download thermal images from the Mendeley dataset (link above)
- Accept the Llama-3.2-Vision license at https://huggingface.co/meta-llama/Llama-3.2-11B-Vision-Instruct
- Have a HuggingFace access token ready

## 0. GPU Check

In [ ]:
!nvidia-smi

## 1. Install Dependencies

In [ ]:
!pip install -q transformers accelerate torch qwen-vl-utils bitsandbytes pillow

## 2. HuggingFace Login

Required for accessing the gated Llama-3.2-Vision model. Make sure you have accepted the license at https://huggingface.co/meta-llama/Llama-3.2-11B-Vision-Instruct first.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Configuration

Verify the paths and filenames below match your downloaded images. One image from each of these 5 fault categories from the Mendeley dataset.

In [ ]:
import os

# verify this path points to your downloaded thermal images
IMAGE_FOLDER = "./thermal_images"

# verify these filenames match what you actually downloaded
IMAGES = [
    ("healthy.jpg",           "Healthy motor"),
    ("short_circuit.jpg",     "Stator short circuit fault"),
    ("stuck_rotor.jpg",       "Stuck rotor"),
    ("fan_failure.jpg",       "Cooling fan failure"),
    ("bearing_fault.jpg",     "Bearing fault"),  # verify 5th category
]

QUESTIONS = [
    "Describe what you see in this thermal image of an industrial motor. Is the equipment operating normally or is there a fault?",
    "Are there any hotspots or abnormal temperature patterns visible? If so, where are they located and what might be causing them?",
    "Based on this thermal image, would you recommend maintenance action? Why or why not?",
]

# quick check that all images are actually there
for fname, label in IMAGES:
    path = os.path.join(IMAGE_FOLDER, fname)
    status = "[OK]" if os.path.isfile(path) else "[MISSING]"
    print(f"{status}  {label:30s}  {path}")

## 4. Preview Test Images

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(1, len(IMAGES), figsize=(4 * len(IMAGES), 4))
for ax, (fname, label) in zip(axes, IMAGES):
    img_path = os.path.join(IMAGE_FOLDER, fname)
    img = Image.open(img_path).convert("RGB")
    ax.imshow(img)
    ax.set_title(label, fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

---
## Part 1: Qwen2.5-VL-7B-Instruct

### 5. Load Qwen Model (4-bit)

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info
import torch

QWEN_MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    QWEN_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
qwen_processor = AutoProcessor.from_pretrained(QWEN_MODEL_NAME)

print("Qwen model loaded (4-bit).")

### 6. Qwen Inference Function

In [ ]:
def ask_qwen(image_path, question):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path},
                {"type": "text", "text": question},
            ],
        }
    ]
    text = qwen_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = qwen_processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    inputs = inputs.to(qwen_model.device)
    output_ids = qwen_model.generate(**inputs, max_new_tokens=512)
    # strip the input tokens so we only get the generated part
    output_ids = [
        out[len(inp):] for inp, out in zip(inputs.input_ids, output_ids)
    ]
    return qwen_processor.batch_decode(output_ids, skip_special_tokens=True)[0]

### 7. Run Qwen on All Images

In [ ]:
qwen_results = {}

for img_file, label in IMAGES:
    img_path = os.path.join(IMAGE_FOLDER, img_file)
    qwen_results[label] = {}
    print(f"\n{'=' * 60}")
    print(f"Image: {label} ({img_file})")
    print(f"{'=' * 60}")
    for i, q in enumerate(QUESTIONS, 1):
        print(f"\n  Q{i}: {q}")
        response = ask_qwen(img_path, q)
        qwen_results[label][f"Q{i}"] = response
        print(f"  A:  {response}")

print("\nDone with Qwen.")

### 8. Save Qwen Results

In [ ]:
import json

QWEN_RESULTS_FILE = "qwen_results.json"

with open(QWEN_RESULTS_FILE, "w") as f:
    json.dump(qwen_results, f, indent=2)

print(f"Saved to {QWEN_RESULTS_FILE}")

### 9. Unload Qwen and Free GPU Memory

In [ ]:
import gc
import torch

del qwen_model
del qwen_processor
gc.collect()
torch.cuda.empty_cache()

print("Qwen unloaded.")
!nvidia-smi

---
## Part 2: Llama-3.2-Vision-11B-Instruct

> If `del model` + `torch.cuda.empty_cache()` didn't fully free VRAM above, restart the runtime and re-run from the Configuration cell (Step 3), then skip Part 1 and continue here.

### 10. Load Llama Model (4-bit)

In [ ]:
import torch
from transformers import MllamaForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from PIL import Image

LLAMA_MODEL_NAME = "meta-llama/Llama-3.2-11B-Vision-Instruct"

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)

llama_model = MllamaForConditionalGeneration.from_pretrained(
    LLAMA_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
llama_processor = AutoProcessor.from_pretrained(LLAMA_MODEL_NAME)

print("Llama model loaded (4-bit).")

### 11. Llama Inference Function

In [ ]:
def ask_llama(image_path, question):
    image = Image.open(image_path).convert("RGB")
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": question},
            ],
        }
    ]
    text = llama_processor.apply_chat_template(
        messages, add_generation_prompt=True
    )
    inputs = llama_processor(
        images=image, text=text, return_tensors="pt"
    ).to(llama_model.device)
    output = llama_model.generate(**inputs, max_new_tokens=512)
    # skip over the input tokens to get just the response
    return llama_processor.decode(
        output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True
    )

### 12. Run Llama on All Images

In [ ]:
llama_results = {}

for img_file, label in IMAGES:
    img_path = os.path.join(IMAGE_FOLDER, img_file)
    llama_results[label] = {}
    print(f"\n{'=' * 60}")
    print(f"Image: {label} ({img_file})")
    print(f"{'=' * 60}")
    for i, q in enumerate(QUESTIONS, 1):
        print(f"\n  Q{i}: {q}")
        response = ask_llama(img_path, q)
        llama_results[label][f"Q{i}"] = response
        print(f"  A:  {response}")

print("\nDone with Llama.")

### 13. Save Llama Results

In [ ]:
import json

LLAMA_RESULTS_FILE = "llama_results.json"

with open(LLAMA_RESULTS_FILE, "w") as f:
    json.dump(llama_results, f, indent=2)

print(f"Saved to {LLAMA_RESULTS_FILE}")

### 14. Unload Llama and Free GPU Memory

In [ ]:
import gc
import torch

del llama_model
del llama_processor
gc.collect()
torch.cuda.empty_cache()

print("Llama unloaded.")
!nvidia-smi

---
## Part 3: Side-by-Side Comparison

### 15. Load Saved Results

In [ ]:
import json

with open("qwen_results.json", "r") as f:
    qwen_results = json.load(f)

with open("llama_results.json", "r") as f:
    llama_results = json.load(f)

print(f"Loaded Qwen results for {len(qwen_results)} images.")
print(f"Loaded Llama results for {len(llama_results)} images.")

### 16. Side-by-Side Output Comparison

In [ ]:
for label in qwen_results:
    print(f"\n{'#' * 70}")
    print(f"  IMAGE: {label}")
    print(f"{'#' * 70}")
    for qkey in sorted(qwen_results[label].keys()):
        q_idx = int(qkey[1:])
        print(f"\n  {qkey}: {QUESTIONS[q_idx - 1]}")
        print(f"  {'~' * 60}")
        print(f"  [Qwen2.5-VL]")
        print(f"  {qwen_results[label][qkey]}")
        print()
        print(f"  [Llama-3.2-Vision]")
        print(f"  {llama_results.get(label, {}).get(qkey, 'N/A')}")
        print()

### 17. Scoring Table

Fill in the scores (1-5) and notes for each image x question pair after reviewing the outputs above.

**Scoring rubric:**
- **5** - Excellent: Accurate identification, specific details, actionable advice
- **4** - Good: Mostly correct, minor omissions
- **3** - Adequate: Partially correct, vague or generic
- **2** - Poor: Mostly incorrect or irrelevant
- **1** - Failure: Completely wrong or refused to answer

In [ ]:
import pandas as pd

rows = []
for _, label in IMAGES:
    for i in range(1, len(QUESTIONS) + 1):
        rows.append({
            "Image": label,
            "Question": f"Q{i}",
            "Qwen_Score": None,   # verify (1-5)
            "Llama_Score": None,  # verify (1-5)
            "Notes": "",          # verify
        })

score_df = pd.DataFrame(rows)
score_df

### 18. Record Scores

Uncomment and set scores per row after reviewing the model outputs above.

In [ ]:
# row 0 = Healthy motor, Q1
# score_df.at[0, "Qwen_Score"] = 4
# score_df.at[0, "Llama_Score"] = 3
# score_df.at[0, "Notes"] = "Qwen identified normal operation; Llama was vague"

# ... fill in the rest ...

# score_df

### 19. Summary Statistics

In [ ]:
# needs all scores filled in first
if score_df["Qwen_Score"].notna().all() and score_df["Llama_Score"].notna().all():
    print("=== Average Scores ===")
    print(f"  Qwen2.5-VL-7B:        {score_df['Qwen_Score'].mean():.2f}")
    print(f"  Llama-3.2-Vision-11B:  {score_df['Llama_Score'].mean():.2f}")
    print()

    print("=== Per-Image Average ===")
    per_image = score_df.groupby("Image")[["Qwen_Score", "Llama_Score"]].mean()
    print(per_image.to_string())
    print()

    print("=== Per-Question Average ===")
    per_q = score_df.groupby("Question")[["Qwen_Score", "Llama_Score"]].mean()
    print(per_q.to_string())
else:
    print("Fill in all scores in the table above first.")

### 20. Save Scoring Table

In [ ]:
score_df.to_csv("vlm_comparison_scores.csv", index=False)
print("Scores saved to vlm_comparison_scores.csv")

---
## Conclusions

- Which model performed better overall?
- Which model was better at fault detection vs. maintenance recommendations?
- Any notable differences in response style or specificity?
- Limitations of this 4-bit quantized comparison?